# Phase VII — Full ArtBench-10 confirmatory analysis (robust download)

This launcher is designed for the public `painting-geometry` repository and fixes the ArtBench download problem seen with `wget`.

It:
- mounts Google Drive;
- clones the repository;
- installs dependencies;
- runs the full resumable Phase VII pipeline;
- uses persistent Drive checkpoints;
- downloads ArtBench through Kaggle single-file candidates first, then a browser-like resumable `requests` fallback to the official host;
- automatically downloads the compact final results ZIP when complete.

Run **Runtime → Run all**.


In [ ]:
# 0. Mount Drive, clone the public repo, install dependencies
import os, sys, subprocess, shutil
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive")

REPO_URL = "https://github.com/ardominguezm/painting-geometry.git"
BRANCH = "multiscale-corpus-analysis"
REPO_DIR = Path("/content/painting-geometry")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
    text=True,
    capture_output=True,
)
if clone.returncode != 0:
    print(clone.stderr)
    raise RuntimeError(
        "Git clone failed. If the repo was just changed from private to public, "
        "wait ~30 seconds, refresh Colab, and rerun."
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ordpy>=1.2.0", "kagglehub", "requests"],
    check=True,
)

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()

print("Repository cloned ✓")
print("Branch:", BRANCH)
print("Commit:", commit)


## 1. Run the complete resumable pipeline

The longest step is full B90/G44 extraction. Checkpoints are written to Google Drive every 500 paintings.

If the runtime disconnects, reopen this notebook and use **Run all** again. Completed chunks are reused.

The ArtBench archive itself is cached in Drive after a successful download.


In [ ]:
# 1. Full Phase VII pipeline
DRIVE_ROOT = Path("/content/drive/MyDrive/painting_geometry_phase7_full")

cmd = [
    sys.executable, "-u",
    str(REPO_DIR / "scripts" / "run_phase7_full_pipeline.py"),
    "--repo-dir", str(REPO_DIR),
    "--drive-root", str(DRIVE_ROOT),
    "--feature-chunk-size", "500",
    "--ordinal-checkpoint-every", "5000",
    "--n-permutations", "4999",
    "--n-bootstrap", "5000",
]

print("Starting Phase VII...")
print("Persistent root:", DRIVE_ROOT)

proc = subprocess.run(cmd)

if proc.returncode != 0:
    raise RuntimeError(
        f"Phase VII stopped with exit code {proc.returncode}. "
        "The detailed traceback is shown immediately above. "
        "Persistent checkpoints in Drive are preserved."
    )

print("Phase VII complete ✓")


## 2. Download the compact final results archive

The full feature matrices remain in Google Drive because they can be large.


In [ ]:
# 2. Download compact results
LIGHT = DRIVE_ROOT / "painting_geometry_phase7_full_results_LIGHT.zip"
FEATURES_ZIP = DRIVE_ROOT / "painting_geometry_phase7_feature_matrices.zip"

if not LIGHT.exists():
    raise FileNotFoundError(LIGHT)

print("Compact results:", LIGHT, f"{LIGHT.stat().st_size/1e6:.1f} MB")
if FEATURES_ZIP.exists():
    print("Full feature archive (kept in Drive):", FEATURES_ZIP, f"{FEATURES_ZIP.stat().st_size/1e6:.1f} MB")

files.download(str(LIGHT))
